# F · Recommendations — from findings to actions

Every recommendation carries **evidence** (which notebook, which number), an **energy range** (never a single
point — the spread between methods is real uncertainty), a **confidence** level, and **how to verify** it.
Money uses the declared tariff assumption in `registry.CONVENTIONS` (THB/kWh, adjustable).

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="obj.round has no effect")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# `pip install -e .` from the repo root makes `sbanalytics` importable; this fallback works without it.
try:
    import sbanalytics
except ImportError:
    import sys
    from pathlib import Path
    sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sbanalytics").is_dir()) / "src"))

from sbanalytics import data as D, registry as R, viz
viz.setup()
pd.set_option("display.precision", 2)
pd.set_option("display.width", 160)
from sbanalytics import baseline as B, anomaly as AN, analysis as A, kpis as K, config as C

In [2]:
h = D.load_hourly(None)
fh = B.floor_hourly(h)
fd = B.floor_daily(fh)
OFFICE = [2, 3, 4, 5, 6, 7]     # F1 has no AC schedule (eda/00 §5): excluded from schedule-based measures

TARIFF = R.conv("tariff_thb")
days = fd[fd.ok].groupby("floor").size()
year = 365 / days                      # annualisation factor per floor
recs = []

## 1 · Raise cooling setpoints (overcooling)

Two independent estimates of how much AC energy one degree is worth, both from `analysis.py` (built for F5, now run per floor):

* **physics** — thermal model calibrated on the 12:00 AC shutdown (needs a lunch shutdown; F2 has none);
* **regression** — AC energy on indoor temperature with weather and calendar controls.

A physics estimate is used only if its confidence interval is finite and not wider than 5× the estimate (on F4 and F7 the capacitance is not identifiable).

In [3]:
import warnings
rows = []
for f in OFFICE:
    hf = h[h.floor == f]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        phys, reg = A.fit_physics(hf, floor=f), A.run_regression(hf, floor=f)
    est = A.estimator_range(phys, reg)
    keep = []
    for _, e in est[est.estimator != "Midpoint"].iterrows():
        width = abs(e.hi - e.lo)
        reliable = np.isfinite(width) and e.pct_per_c < 0 and width <= 5 * abs(e.pct_per_c)
        rows.append({"floor": f, "estimator": e.estimator.split(" —")[0].split(" on")[0], "pct_per_c": e.pct_per_c,
                     "ci_lo": e.lo, "ci_hi": e.hi, "used": reliable})
        if reliable:
            keep.append(-e.pct_per_c)
sens = pd.DataFrame(rows)
sens.round(2)

,floor,estimator,pct_per_c,ci_lo,ci_hi,used
0,2,Regression,-5.74,-6.58e+00,-4.90,True
1,3,Physics,-2.36,-4.61e+00,-1.33,True
2,3,Regression,-2.35,-3.50e+00,-1.19,True
3,4,Physics,-8.33,-1.56e+09,-3.29,False
4,4,Regression,-6.17,-7.05e+00,-5.28,True
5,5,Physics,-3.92,-1.21e+01,-1.89,True
6,5,Regression,-7.69,-8.93e+00,-6.45,True
7,6,Physics,-4.53,-1.30e+01,-2.36,True
8,6,Regression,-3.47,-4.61e+00,-2.33,True
9,7,Physics,-27.97,-1.35e+09,-5.29,False


In [4]:
rng = sens[sens.used].groupby("floor").pct_per_c.agg(lambda s: (-s.max(), -s.min()))
setpt = []
for f in OFFICE:
    if f not in rng.index:
        continue
    lo_pct, hi_pct = rng[f]
    hf = h[h.floor == f]
    base = A.setpoint_saving(hf, pct_per_c=-1, d_temp=1, floor=f)      # annual occupied AC kWh per zone
    kwh_year = base.kwh_year.sum()
    n_est = int(sens[(sens.floor == f) & sens.used].shape[0])
    setpt.append({"floor": f, "occupied_ac_kwh_year": kwh_year, "pct_per_c_low": lo_pct, "pct_per_c_high": hi_pct,
                  "saving_kwh_low_per_c": kwh_year * lo_pct / 100, "saving_kwh_high_per_c": kwh_year * hi_pct / 100,
                  "estimators_agreeing": n_est})
setpt = pd.DataFrame(setpt).set_index("floor")
setpt.round(1)

,occupied_ac_kwh_year,pct_per_c_low,pct_per_c_high,saving_kwh_low_per_c,saving_kwh_high_per_c,estimators_agreeing
floor,,,,,,
2,134587.8,5.7,5.7,7721.3,7721.3,1
3,102389.8,2.3,2.4,2401.5,2421.2,2
4,113625.3,6.2,6.2,7008.3,7008.3,1
5,73960.6,3.9,7.7,2895.7,5690.8,2
6,91756.4,3.5,4.5,3185.1,4157.4,2
7,170443.0,1.4,1.4,2453.5,2453.5,1


In [5]:
oc = pd.read_csv(C.DATA_DERIVED / "floor_kpis.csv", index_col="floor").overcooled_share
for d_t in (1, 2):
    lo, hi = setpt.saving_kwh_low_per_c.sum() * d_t, setpt.saving_kwh_high_per_c.sum() * d_t
    print(f"+{d_t} °C on floors {list(setpt.index)}: {lo:,.0f}–{hi:,.0f} kWh/yr  ({lo*TARIFF:,.0f}–{hi*TARIFF:,.0f} THB/yr)")
lo, hi = setpt.saving_kwh_low_per_c.sum() * 2, setpt.saving_kwh_high_per_c.sum() * 2
conf = "medium" if (setpt.estimators_agreeing >= 2).sum() >= 2 else "low"
recs.append({"action": "Raise occupied cooling setpoint by 2 °C (towards 24–25 °C) on office floors",
             "floors": ", ".join(f"F{f}" for f in setpt.index), "kwh_per_year_low": lo, "kwh_per_year_high": hi,
             "confidence": conf,
             "evidence": f"{oc.loc[OFFICE].min():.0%}–{oc.loc[OFFICE].max():.0%} of occupied cooled hours below 23 °C (A); "
                         f"{setpt.pct_per_c_low.min():.1f}–{setpt.pct_per_c_high.max():.1f}% AC energy per °C",
             "verify": "Pilot on one floor; compare against its G14 baseline (B) for the weeks given in §6"})

+1 °C on floors [2, 3, 4, 5, 6, 7]: 25,665–29,452 kWh/yr  (107,795–123,700 THB/yr)
+2 °C on floors [2, 3, 4, 5, 6, 7]: 51,331–58,905 kWh/yr  (215,589–247,401 THB/yr)


## 2 · Lock AC schedules to the calendar (after-hours and weekend cooling)

In [6]:
ah = AN.after_hours_ac(fh[fh.floor.isin(OFFICE)])
ah["weekend_or_holiday"] = ah.day_type.isin(["weekend", "holiday", "closure"])
per = ah.groupby(["floor", "weekend_or_holiday"]).excess_kwh.sum().unstack(fill_value=0)
per = per.mul(year.loc[per.index], axis=0).rename(columns={True: "weekend/holiday kWh/yr", False: "workday out-of-hours kWh/yr"})
per["THB/yr"] = per.sum(axis=1) * TARIFF
display(per.round(0))
# Not all of it is avoidable: some runs are authorised (exams, events). Range = half to all of the weekend/holiday excess.
wk = per["weekend/holiday kWh/yr"].sum()
recs.append({"action": "Tie AC start/stop to the holiday calendar and require sign-off for weekend runs",
             "floors": ", ".join(f"F{f}" for f in per.index), "kwh_per_year_low": 0.5 * wk, "kwh_per_year_high": wk,
             "confidence": "high", "evidence": "after-hours AC events above each floor's own night baseline (D §2); "
                                              "workday-shaped weekend days (A)",
             "verify": "Count of weekend AC events per month in the fault register (D) should fall"})
wd = per["workday out-of-hours kWh/yr"].sum()
recs.append({"action": "Trim AC run-on after the scheduled stop on workdays",
             "floors": ", ".join(f"F{f}" for f in per.index), "kwh_per_year_low": 0.5 * wd, "kwh_per_year_high": wd,
             "confidence": "medium", "evidence": "workday after-hours AC events (D §2)",
             "verify": "Hourly profile after 17:00 against baseline (B)"})

weekend_or_holiday,workday out-of-hours kWh/yr,weekend/holiday kWh/yr,THB/yr
floor,,,
2,4188.0,5492.0,40655.0
3,123.0,1351.0,6190.0
4,2.0,2397.0,10076.0
5,38.0,722.0,3193.0
6,62.0,669.0,3069.0
7,4756.0,12065.0,70646.0


## 3 · Always-on AC units

In [7]:
hp = D.load_hourly_points()
units = AN.unit_status(hp)
on = units[units.status == "always-on"]
key = on[["floor", "column"]].astype({"column": str})
night = hp[hp.ts.dt.hour.between(1, 4)].astype({"column": str}).merge(key, on=["floor", "column"])   # match floor AND column
night_kwh = (night.groupby(["floor", "column"]).value.sum()
             .reset_index().assign(kwh_per_year=lambda x: x.value * x.floor.map(year)))
display(on.astype({"column": str}).merge(night_kwh[["floor", "column", "kwh_per_year"]], on=["floor", "column"]).round(3))
office_on = night_kwh[night_kwh.floor.isin(OFFICE)]
recs.append({"action": "Confirm whether always-on AC units serve 24/7 rooms; if not, schedule them off overnight",
             "floors": ", ".join(sorted({f"F{f}" for f in office_on.floor})), "kwh_per_year_low": 0.0,
             "kwh_per_year_high": office_on.kwh_per_year.sum(),
             "confidence": "low", "evidence": "units running in >80% of night hours (eda/00 §4); range covers 01–05 h only",
             "verify": "Unit's night kWh in the hourly point table"})
f1 = night_kwh[night_kwh.floor == 1]
recs.append({"action": "Floor 1: establish what the space is and whether its 24/7 cooling is required (outside the office-floor total)",
             "floors": "F1", "kwh_per_year_low": 0.0, "kwh_per_year_high": f1.kwh_per_year.sum(),
             "confidence": "low", "evidence": "z2_AC1 runs in 99% of night hours; F1 is not an office profile (eda/00 §5)",
             "verify": "Site visit; then baseline F1 separately"})

,floor,zone,column,role,kwh,hours,on_share,night_on_share,floor_share,status,kwh_per_year
0,1,z2,z2_AC1(kW),primary,368478.88,13152,0.78,0.99,0.85,always-on,50736.96
1,2,z2,z2_AC1(kW),primary,77045.85,12677,0.94,1.00,0.27,always-on,1132.13
2,2,z2,z2_AC6(kW),primary,7710.68,13165,0.79,1.00,0.03,always-on,1361.11


## 4 · Overnight baseload (lighting and plug)

Workday nights 01:00–04:59: lighting + plug energy above each floor's own 5th-percentile night — the level the floor demonstrably reaches when things are switched off.

In [8]:
n = fh[fh.floor.isin(OFFICE) & fh.ok & fh.hour.between(1, 4)].copy()
n["lp"] = n.light_kwh + n.plug_kwh
p5 = n.groupby("floor").lp.quantile(0.05)
excess = (n.lp - n.floor.map(p5)).clip(lower=0).groupby(n.floor).sum() * year.loc[OFFICE]
base = pd.DataFrame({"night light+plug kWh/h (mean)": n.groupby("floor").lp.mean(), "P5 kWh/h": p5,
                     "excess kWh/yr (01-05 h)": excess})
display(base.round(2))
recs.append({"action": "Night switch-off sweep for lighting and plug loads (timers / smart plugs)",
             "floors": ", ".join(f"F{f}" for f in OFFICE), "kwh_per_year_low": 0.5 * excess.sum(),
             "kwh_per_year_high": excess.sum() * 24 / 4 * 0.5,
             "confidence": "low", "evidence": "night load above each floor's own P5 night (01–05 h measured; upper end extrapolates to half of all off-hours)",
             "verify": "Night baseload ratio (A) before/after"})

,night light+plug kWh/h (mean),P5 kWh/h,excess kWh/yr (01-05 h)
floor,,,
2,0.87,0.59,404.81
3,0.67,0.50,260.51
4,1.09,0.64,685.39
5,0.90,0.69,311.26
6,0.92,0.69,334.89
7,0.50,0.16,491.36


## 5 · Data and metering actions (no direct kWh, but every other number depends on them)

In [9]:
dq_actions = [
    {"action": "Fix the BMS sensor logging gap (no indoor temperature/RH on any floor for ~6 months, Sep 2018 – Mar 2019)",
     "floors": "all", "confidence": "high", "evidence": "eda/00 §2 coverage map", "verify": "Monthly completeness ≥ 95%"},
    {"action": "Restore the F6 data feed (export ends 22 Oct 2019 mid-row)", "floors": "F6", "confidence": "high",
     "evidence": "eda/00 §1", "verify": "File covers full period"},
    {"action": "Confirm Floor 1's use and logger clock (evening-peaking, AC never off; 7 h clock offset not ruled out)",
     "floors": "F1", "confidence": "medium", "evidence": "eda/00 §5", "verify": "Compare logger time against a known switch event"},
    {"action": "Inspect dead / never-running AC meters and frozen plug meters; confirm the candidate meter re-assignments",
     "floors": "F2, F5, F7 (+ candidates)", "confidence": "high",
     "evidence": "eda/00 §4, §7; D §1", "verify": "Point health score (D) back above 95"},
]
for a in dq_actions:
    recs.append({**a, "kwh_per_year_low": np.nan, "kwh_per_year_high": np.nan})

## 6 · How long a pilot needs to run to prove it

Minimum detectable saving for a before/after comparison against the baseline, at α = 0.05 and 80 % power, using each floor's out-of-sample hourly CV(RMSE) from B (`analysis.weeks_required`). Hourly residuals are autocorrelated; ρ = 0.5 within a day is assumed, which lengthens the pilot.

In [10]:
g14 = pd.read_csv(C.DATA_DERIVED / "baseline_g14.csv")
cv = (g14[(g14.target == "total_kwh") & g14.split.str.startswith("test") & (g14.model == "TOWT")]
      .set_index("floor").cv_rmse)
pilot = pd.DataFrame({"hourly CV(RMSE) %": cv,
                      **{f"weeks to detect {p}%": [A.weeks_required(c, p, hours_per_day=24, days_per_week=7, rho=0.5) for c in cv]
                         for p in (3, 5, 10)}})
pilot.round(1)

,hourly CV(RMSE) %,weeks to detect 3%,weeks to detect 5%,weeks to detect 10%
floor,,,,
1,28.6,54,20,5
2,29.4,56,21,6
3,24.3,39,14,4
4,22.7,34,13,4
5,22.2,33,12,3
6,19.7,26,10,3
7,35.4,82,30,8


## 7 · The recommendation table

In [11]:
out = pd.DataFrame(recs)
out["thb_per_year_low"] = out.kwh_per_year_low * TARIFF
out["thb_per_year_high"] = out.kwh_per_year_high * TARIFF
out = out[["action", "floors", "kwh_per_year_low", "kwh_per_year_high", "thb_per_year_low", "thb_per_year_high",
           "confidence", "evidence", "verify"]]
office_rows = ~out.floors.str.contains("F1")
total_lo, total_hi = out[office_rows].kwh_per_year_low.sum(), out[office_rows].kwh_per_year_high.sum()
metered = fd[fd.ok & fd.floor.isin(OFFICE)].groupby("floor").total_kwh.sum().mul(year.loc[OFFICE]).sum()
print(f"Quantified opportunities, office floors: {total_lo:,.0f}–{total_hi:,.0f} kWh/yr "
      f"({total_lo/metered:.0%}–{total_hi/metered:.0%} of their metered energy), "
      f"{total_lo*TARIFF:,.0f}–{total_hi*TARIFF:,.0f} THB/yr at {TARIFF} THB/kWh. Ranges overlap partly (a setpoint change also cuts after-hours AC), so do not add the high ends.")
out.style.format({c: "{:,.0f}" for c in out.columns if "per_year" in c}, na_rep="—")

Quantified opportunities, office floors: 68,507–100,727 kWh/yr (5%–8% of their metered energy), 287,729–423,053 THB/yr at 4.2 THB/kWh. Ranges overlap partly (a setpoint change also cuts after-hours AC), so do not add the high ends.


,action,floors,kwh_per_year_low,kwh_per_year_high,thb_per_year_low,thb_per_year_high,confidence,evidence,verify
0,Raise occupied cooling setpoint by 2 °C (towards 24–25 °C) on office floors,"F2, F3, F4, F5, F6, F7","51,331","58,905","215,589","247,401",medium,68%–83% of occupied cooled hours below 23 °C (A); 1.4–7.7% AC energy per °C,Pilot on one floor; compare against its G14 baseline (B) for the weeks given in §6
1,Tie AC start/stop to the holiday calendar and require sign-off for weekend runs,"F2, F3, F4, F5, F6, F7","11,348","22,696","47,662","95,324",high,after-hours AC events above each floor's own night baseline (D §2); workday-shaped weekend days (A),Count of weekend AC events per month in the fault register (D) should fall
2,Trim AC run-on after the scheduled stop on workdays,"F2, F3, F4, F5, F6, F7","4,584","9,168","19,252","38,505",medium,workday after-hours AC events (D §2),Hourly profile after 17:00 against baseline (B)
3,"Confirm whether always-on AC units serve 24/7 rooms; if not, schedule them off overnight",F2,0,"2,493",0,"10,472",low,units running in >80% of night hours (eda/00 §4); range covers 01–05 h only,Unit's night kWh in the hourly point table
4,Floor 1: establish what the space is and whether its 24/7 cooling is required (outside the office-floor total),F1,0,"50,737",0,"213,095",low,z2_AC1 runs in 99% of night hours; F1 is not an office profile (eda/00 §5),Site visit; then baseline F1 separately
5,Night switch-off sweep for lighting and plug loads (timers / smart plugs),"F2, F3, F4, F5, F6, F7","1,244","7,465","5,225","31,352",low,night load above each floor's own P5 night (01–05 h measured; upper end extrapolates to half of all off-hours),Night baseload ratio (A) before/after
6,"Fix the BMS sensor logging gap (no indoor temperature/RH on any floor for ~6 months, Sep 2018 – Mar 2019)",all,—,—,—,—,high,eda/00 §2 coverage map,Monthly completeness ≥ 95%
7,Restore the F6 data feed (export ends 22 Oct 2019 mid-row),F6,—,—,—,—,high,eda/00 §1,File covers full period
8,"Confirm Floor 1's use and logger clock (evening-peaking, AC never off; 7 h clock offset not ruled out)",F1,—,—,—,—,medium,eda/00 §5,Compare logger time against a known switch event
9,Inspect dead / never-running AC meters and frozen plug meters; confirm the candidate meter re-assignments,"F2, F5, F7 (+ candidates)",—,—,—,—,high,"eda/00 §4, §7; D §1",Point health score (D) back above 95
